# Day 31 — Practice Questions
### What Machine Learning Actually Is

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — no computer needed | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A is pen-and-paper first.** Write your answer, *then* run the cell to check.

Three questions to keep asking yourself:

- Is this **supervised** or **unsupervised**? If supervised, **number** or **group**?
- Does this column have a **real order**, or am I inventing one?
- Is this score measured on rows the model has **already seen**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

---
# Part A — Work it out

Eight questions. Most need no computer at all.

### A1 · Read the table

Below is a small dataset of second-hand scooters. Before running anything, write down:

1. How many **samples** are there?
2. How many **features**?
3. Which column is the **label**, if we want to predict the price?

In [ ]:
scooters = pd.DataFrame({
    "brand":    ["Honda", "TVS", "Bajaj", "Honda", "Hero"],
    "age_yr":   [3, 1, 5, 2, 4],
    "km_run":   [18000, 4200, 31000, 11500, 25000],
    "mileage":  [45, 52, 40, 45, 48],
    "price_k":  [52, 78, 31, 61, 42],
})
scooters

### A2 · Name the problem

For each of these, say: supervised or unsupervised? And if supervised, number or group?

| | The problem |
|---|---|
| a | Predict a scooter's resale price from its age and kilometres |
| b | Decide whether a student will pass or fail |
| c | Split 5,000 Instagram users into groups by what they post |
| d | Predict how many minutes a train will be delayed |
| e | Decide which of 10 diseases a scan shows |

### A3 · Count the columns

If we one-hot encode the `brand` column in the scooter table, how many columns will the
whole table have afterwards? Work it out before running.

In [ ]:
encoded = pd.get_dummies(scooters, columns=["brand"], dtype=int)
print("before:", scooters.shape[1], "columns")
print("after :", encoded.shape[1], "columns")
print()
print(list(encoded.columns))

### A4 · Which of these columns has a real order?

For each one, say whether you would **one-hot encode** it or **just number it**.

| | The column | Values |
|---|---|---|
| a | `blood_group` | A+, B+, O+, AB+ |
| b | `t_shirt_size` | S, M, L, XL |
| c | `city` | Mumbai, Delhi, Chennai |
| d | `satisfaction` | Poor, Average, Good, Excellent |
| e | `has_licence` | yes, no |

### A5 · Spot what the numbering is claiming

Someone encoded `city` as Chennai=0, Delhi=1, Mumbai=2.

Write down **three things** the model now believes that are not true.

In [ ]:
cities = {"Chennai": 0, "Delhi": 1, "Mumbai": 2}
print(cities)
print()
print("the model can now do arithmetic with these. What does it conclude?")

### A6 · Which score should you report?

A classmate shows you this and says the model is 94% accurate.

In [ ]:
rng = np.random.default_rng(55)
x = rng.uniform(0, 10, 40)
y = 20 + 3*x + rng.normal(0, 5, 40)

X_study, X_hidden, y_study, y_hidden = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0)

tree = DecisionTreeRegressor(random_state=0).fit(X_study, y_study)

print("score on the rows it studied :", round(tree.score(X_study, y_study), 3))
print("score on the HIDDEN rows     :", round(tree.score(X_hidden, y_hidden), 3))

### A7 · What does the error tell you?

Our model guesses laptop prices. Here are its five guesses next to the truth. Work out the
total squared error by hand, then check.

In [ ]:
truth   = np.array([30., 45., 52., 38., 61.])
guesses = np.array([32., 42., 52., 40., 59.])

errors = truth - guesses
print("the errors        :", errors)
print("squared           :", errors ** 2)
print("added up          :", (errors ** 2).sum())

### A8 · How many columns is too many?

A delivery dataset has 2,000 rows and a `restaurant_name` column with 1,350 different
restaurants in it.

What happens if you one-hot encode it? Answer in words before running.

In [ ]:
rows = 2000
distinct_restaurants = 1350

print("rows                      :", rows)
print("columns after one-hot     :", distinct_restaurants)
print("rows per column (average) :", round(rows / distinct_restaurants, 2))

---
# Part B — Find the mistake

Four short programs. Each one either crashes or gives a misleading answer. Find it, say
what is wrong in one sentence, then fix it.

### B1 · This one crashes

In [ ]:
cars = pd.DataFrame({
    "fuel":   ["petrol", "diesel", "petrol", "cng"],
    "age_yr": [3, 5, 2, 7],
    "price":  [420, 310, 510, 180],
})

X = cars[["fuel", "age_yr"]]
y = cars["price"]

try:
    LinearRegression().fit(X, y)
except Exception as e:
    print(type(e).__name__, ":", e)

### B2 · It runs, and the answer is wrong

In [ ]:
rng = np.random.default_rng(77)
N = 400

city  = rng.choice(["Chennai", "Delhi", "Mumbai"], N)
extra = pd.Series(city).map({"Chennai": 0, "Delhi": 5, "Mumbai": 40}).values
size  = rng.integers(400, 1600, N)
rent  = 3000 + 12*size + 400*extra + rng.normal(0, 1500, N)

homes = pd.DataFrame({"city": city, "size_sqft": size, "rent": rent})

# the analyst numbers the cities alphabetically
numbering = {"Chennai": 0, "Delhi": 1, "Mumbai": 2}
bad = homes.copy()
bad["city"] = bad["city"].map(numbering)          # <-- look here

X_study, X_hidden, y_study, y_hidden = train_test_split(
    bad.drop(columns=["rent"]), bad["rent"], test_size=.3, random_state=0)
m = LinearRegression().fit(X_study, y_study)
print("average error : ₹", round(mean_absolute_error(y_hidden, m.predict(X_hidden))))

### B3 · A suspiciously good model

In [ ]:
marks = pd.DataFrame({
    "hours":      [2, 4, 5, 7, 8, 3, 6, 9, 1, 5],
    "attendance": [60, 75, 80, 90, 95, 65, 85, 98, 50, 78],
    "final_mark": [41, 55, 62, 74, 81, 48, 68, 88, 35, 60],
})

X = marks                            # <-- look here
y = marks["final_mark"]

X_study, X_hidden, y_study, y_hidden = train_test_split(X, y, test_size=.3, random_state=0)
m = LinearRegression().fit(X_study, y_study)
print("score on hidden rows:", round(m.score(X_hidden, y_hidden), 6))

### B4 · The score changes every time

In [ ]:
rng = np.random.default_rng(99)
x = rng.uniform(0, 10, 50)
y = 10 + 2*x + rng.normal(0, 4, 50)

for attempt in range(4):
    a, b, c, d = train_test_split(x.reshape(-1,1), y, test_size=.3)   # <-- look here
    m = LinearRegression().fit(a, c)
    print(f"run {attempt+1}:  score = {m.score(b, d):.4f}")

---
# Part C — Write the code

Five tasks on one dataset. Write your answer in the empty cell; a worked solution follows.

In [ ]:
# ---------------------------------------------------------------
# 700 second-hand phones. Run this once - tasks C1 to C5 use `phones`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3131)
n = 700

brand   = rng.choice(["Apple", "Samsung", "Xiaomi", "Realme"], n, p=[.22, .30, .26, .22])
premium = pd.Series(brand).map({"Apple": 38, "Samsung": 12, "Xiaomi": 2, "Realme": 0}).values
condition = rng.choice(["Poor", "Fair", "Good", "Excellent"], n)
cond_bump = pd.Series(condition).map({"Poor": 0, "Fair": 6, "Good": 12, "Excellent": 18}).values
ram     = rng.choice([4, 6, 8, 12], n)
age_mo  = rng.integers(1, 48, n)

price = (8 + 1.6*ram + premium + cond_bump - 0.22*age_mo + rng.normal(0, 3, n))

phones = pd.DataFrame({"brand": brand, "condition": condition, "ram_gb": ram,
                       "age_months": age_mo, "price_k": price.round(1)})
phones.head()

### C1 · Describe the dataset

Print, in plain language:

1. how many samples there are
2. the names of the features
3. the name of the label (we want to predict `price_k`)
4. which columns are words and will need encoding

In [ ]:
# your code here

### C2 · Encode the two word columns — each the right way

Build a new table where:

- `brand` is one-hot encoded
- `condition` is numbered 1 (Poor) to 4 (Excellent)

Print how many columns you end up with.

In [ ]:
# your code here

### C3 · Train and score it honestly

Hide 30% of the rows (`random_state=0`), train a `LinearRegression`, and report:

- the score on the hidden rows
- the score on the studied rows
- the average error in thousands of rupees

In [ ]:
# your code here

### C4 · Look inside the model

Print each input column next to the weight the model learned for it.

Then answer in a comment: **which brand is worth the most**, and does the model agree with
how the data was built?

In [ ]:
# your code here

### C5 · Prove the encoding mattered

Build a **second** version where `brand` is numbered alphabetically instead of one-hot
encoded, and compare the two on hidden rows.

In [ ]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Make a broken dataset trainable

The table below cannot be fed to any model as it stands. There are **four** separate
problems with it.

**Your tasks:**

1. Find all four. Write each one down before you fix anything.
2. Fix them, in a sensible order.
3. Train a model and report an honest score.
4. Say in a comment which fix you think mattered most, and why.

In [ ]:
rng = np.random.default_rng(4141)
n = 500

area   = rng.integers(500, 2500, n)
loc    = rng.choice(["Andheri", "Bandra", "Dadar", "Thane", "Powai"], n)
loc_b  = pd.Series(loc).map({"Andheri": 8, "Bandra": 30, "Dadar": 14,
                             "Thane": 0, "Powai": 18}).values
furn   = rng.choice(["Unfurnished", "Semi", "Full"], n)
furn_b = pd.Series(furn).map({"Unfurnished": 0, "Semi": 7, "Full": 15}).values
pin    = rng.integers(400001, 400001 + 300, n).astype(str)

rent = 5000 + 22*area + 900*loc_b + 700*furn_b + rng.normal(0, 2500, n)

flats = pd.DataFrame({
    "area_sqft":   area,
    "locality":    loc,
    "furnishing":  furn,
    "pincode":     pin,
    "rent":        rent.round(0),
    "rent_in_usd": (rent / 83).round(2),       # the same rent, in dollars
})
flats.head()

In [ ]:
# your investigation here

### D2 · Write a reusable checker

Write a function `check_dataset(df, label)` that looks at any table and prints a short
report before anyone trains anything on it.

It must report:

1. how many samples and how many features
2. which columns are words, and how many different values each has
3. a **warning** for any word column with more than 20 different values
4. a **warning** for any column that is suspiciously close to a copy of the label
5. a suggested encoding for each word column, based on how many values it has

Then run it on `flats` and show that it catches all four problems from D1.

In [ ]:
# your code here

---
## Before you hand it in

- Did you **encode every word column** before training?
- For each one: does it have a **real order**, or did you invent one?
- Did you **drop the label** from the inputs?
- Is your score measured on rows the model has **never seen**?
- Did you set `random_state` so your result repeats?
- If any score looked **perfect**, did you go and find out why?

Three numbers worth carrying away:

- Numbering brands instead of one-hot encoding them **tripled the error**: ₹3.3k → ₹10.2k.
- A tree scored **1.000 on rows it studied and 0.644 on new ones**. Only the second is real.
- One-hot encoding 426 pincodes from 900 rows gave a score of **−0.494** — worse than
  guessing.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*